In [1]:
import warnings
warnings.filterwarnings('ignore')

# Imports

In [2]:
import torch
import scanpy as sc
import numpy as np
import anndata as ad
from tqdm import tqdm
import os

from scdisentangle.train.tools import get_trainer, set_seed

# Params

In [3]:
seed_nb = 42
yaml_path = '../configs/leukemia_disentangle.yaml'
weights_path =  '../weights/MIG_BINNED_dis_latent_stack_Sample_id_train'

counterfactual_dict = {
    'Sample_id': 'patient1_IP'
}

covariate_name = 'cell_type'

# Set seed

In [4]:
set_seed(seed_nb)

ic| 'Setting seed to', seed: 42


# Get trainer

In [5]:
trainer = get_trainer(yaml_path, wandb_log=False)
trainer.load_weights(weights_path)

Global seed set to 0
ic| 'Setting seed to', seed: 42
ic| 'Creating cell mappings'
ic| 'Creating inputs'
ic| 'Creating inputs'


Wandb is off
Loading weights from ../weights/MIG_BINNED_dis_latent_stack_Sample_id_train


# Predict to get latent

In [6]:
adata = trainer.predict(
    trainer.dataset.data.copy(), 
    counterfactual_dict={}, 
    bs=256
)
adata.layers['org_expression'] = trainer.dataset.data.X.copy()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 145/145 [00:01<00:00, 136.73it/s]


In [7]:
# Filter
adata = adata[adata.obs['cell_type'] != 'Ribosomal/Mitochondrial/Degraded cells'].copy()

In [8]:
adata

AnnData object with n_obs × n_vars = 36175 × 5000
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.ribo', 'percent.mito', 'Sample_id', 'Transduction', 'Phase', 'Timepoint', 'Condition', 'CARexpresion', 'cloneType', 'Frequency', 'author_cell_type', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'is_primary_data', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_counts', 'sc_cell_ids', 'cell_ids', 'split_anndata'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    obsm: 'HARMONY', 'X_UMAP', 'dis_latent_stack', 'cat_latent_stack', 'cat_

# Get progressive latent (and optionally recs)

In [9]:
from scdisentangle.train.progressive_latent import get_progressive_latent

In [10]:
# Balanced
adata_structured_balanced = get_progressive_latent(
    trainer=trainer,
    adata=adata,
    counterfactual_dict=counterfactual_dict,
    get_recs=True,
    balance_clusters=True,
    covariate_name='cell_type',
    )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:02<00:00,  7.21it/s]


In [11]:
# Unbalanced
adata_structured_unbalanced = get_progressive_latent(
    trainer=trainer,
    adata=adata,
    counterfactual_dict=counterfactual_dict,
    get_recs=True,
    balance_clusters=False,
    covariate_name='cell_type',
    )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:07<00:00,  2.10it/s]


# Save adata with latent levels

In [12]:
adata_structured_balanced.write_h5ad('adata_structured_balanced.h5ad')
adata_structured_unbalanced.write_h5ad('adata_structured_unbalanced.h5ad')